# Responses and prompts

This notebook shows what a chat-model response contains, why models count text in tokens, and how prompt templates help us safely reuse a prompt.


In [17]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

# Load GROQ_API_KEY from the local .env file.
load_dotenv()

groq_model = init_chat_model(
    "openai/gpt-oss-120b",
    model_provider="groq",
)


## Inspect a response

`invoke` returns an AI message object. The generated text is only one part of it; metadata and token usage can help when debugging an application.


In [18]:
response = groq_model.invoke("Tell me a joke about AI and humans.")

print("Text:", response.text)
print()
print("Model name:", response.response_metadata.get("model_name"))
print("Model provider:", response.response_metadata.get("model_provider"))
print("Token usage:", response.usage_metadata)


Text: Why did the AI break up with the human?

Because every time the human said, “Let’s *sync* up later,” the AI thought they meant a data sync—so it spent the whole night trying to merge their calendars, while the human was just asking for coffee! ☕🤖😂

Model name: openai/gpt-oss-120b
Model provider: groq
Token usage: {'input_tokens': 80, 'output_tokens': 96, 'total_tokens': 176, 'output_token_details': {'reasoning': 26}}


## Tokens and model names

A token is a small piece of text a model can turn into numbers. It may be a word, part of a word, punctuation, or whitespace. We send input tokens in our instructions, questions, and context; the model sends output tokens back in its answer. Providers normally charge for both.

The model setup also has three names: our Python variable (`groq_model`), the provider integration (`groq`), and the exact model identifier (`openai/gpt-oss-120b`). They are not interchangeable.


In [19]:
usage = response.usage_metadata or {}

# Input tokens are what we send; output tokens are what the model generates.
print("Input tokens:", usage.get("input_tokens", "not reported"))
print("Output tokens:", usage.get("output_tokens", "not reported"))
print("Total tokens:", usage.get("total_tokens", "not reported"))

# Record the provider and exact model when comparing behaviour or cost.
print("Provider:", response.response_metadata.get("model_provider"))
print("Model:", response.response_metadata.get("model_name"))


Input tokens: 80
Output tokens: 96
Total tokens: 176
Provider: groq
Model: openai/gpt-oss-120b


## Static and dynamic prompts

A static prompt is fixed text. A dynamic prompt contains values supplied at runtime, such as a question, a customer name, or content retrieved from a document.


In [23]:
static_prompt = "Will Ash's Pikachu ever evolve ?"
print(groq_model.invoke(static_prompt, max_tokens=200).text)


users_question = "How do you make a sandwich?"  # This is your user input, which can be dynamically changed at runtime.
dynamic_prompt = f"Answer this question like a comedian: {users_question}"  # simple string interpolation to create a dynamic prompt
print("Dynamic Prompt:")
print(groq_model.invoke(dynamic_prompt, max_tokens=200).text)


**Short answer:** No—Ash’s Pikachu has never evolved in the official Pokémon anime, and there are no announced plans for it to evolve in the future.

**Why Pikachu stays the same**

| Reason | Explanation |
|--------|--------------|
| **Signature mascot** | Pikachu is the face of the whole Pokémon franchise (and the mascot of the series itself). Keeping Ash’s Pikachu at its iconic “Pikachu” stage reinforces brand recognition. |
| **Character continuity** | Throughout more than 25 years of episodes, movies, specials, and spin‑offs, Ash’s Pikachu has remained a constant companion. An evolution would dramatically change the dynamic
Dynamic Prompt:
**How to Make a Sandwich—Comedian Edition**

1. **Gather Your Weapons**  
   First, locate the two pieces of bread. If you can’t find them, check the “emergency stash” behind the cereal boxes—because nothing says “I’m an adult” like a stale baguette you’ve been hoarding since 2019.

2. **Pick a Filling**  
   Choose something that won’t judge yo

## Reuse, validate, and save a prompt template

A normal Python string or f-string is fine for a one-off prompt. `PromptTemplate` is useful when the same prompt should be reused and its required values should be checked before we call a model.



In [21]:
from langchain_core.prompts import PromptTemplate

dynamic_question_template = PromptTemplate(
    template="Answer the question as a sarcastic teacher: {question}",
    input_variables=["question"],
)
question = "Why did jack and jill go up the hill?"  # This is your user input, which can be dynamically changed at runtime.
prompt = dynamic_question_template.invoke({"question": question})

result = groq_model.invoke(prompt, max_tokens=200)
print(result.text)


# if requred we can save the above prompt template to a file and load it later for reuse. This is useful for maintaining consistency in prompts across different parts of an application or across different applications.

Ah, of course—because the universe simply *had* to give us a timeless lesson in physics and poor decision‑making. Jack and Jill climbed the hill for the *obviously* noble purpose of proving that gravity works, that water can be fetched, and that teenagers love to ignore safety warnings. In other words, they wanted a spectacularly avoidable tumble so we could all sit around and repeat the rhyme for eternity. Brilliant, right? 🙄
